# Predicción del uso problemático de internet en niños y adolescentes
### MIAA 2026-2 — Notebook único de análisis exploratorio y modelos de referencia

**Entorno:** Google Colab o Jupyter con Python 3.11+.  
**Fuente:** [Child Mind Institute — Problematic Internet Use (Kaggle)](https://www.kaggle.com/competitions/child-mind-institute-problematic-internet-use).  
**Desenlace:** `sii`, categorías ordinales 0, 1, 2 y 3. **Cohorte principal:** 5–17 años; sensibilidades optativas 5–19 y 5–22.  
**Evaluación:** QWK, F1-macro, matrices de confusión, precisión y sensibilidad de `sii=3`.

Este notebook reúne el **EDA del Notebook 03**, el **protocolo de referencia del Notebook 04** y las **ablaciones exploratorias** documentadas en la segunda entrega. No utiliza `PCIAT-*`, `id` ni `sii` como *predictoras*. Se usa `sii` exclusivamente como etiqueta. No implica validación externa, predicción prospectiva ni diagnóstico.

> **Seguridad:** guarda tu token en **Secretos** de Google Colab con el nombre `KAGGLE_API_TOKEN` y habilita el acceso de este notebook. No escribas el token en celdas, archivos del proyecto ni repositorios. Los resultados exportables no incluyen identificadores ni registros individuales.

**Orden recomendado:** Ejecutar todo, de arriba a abajo. El entrenamiento completo puede tardar según el equipo y la versión de las bibliotecas. Las cifras del informe son puntos de referencia históricos, no salidas precargadas.

## 0. Parámetros y dependencias

- `FUENTE_DATOS="KAGGLE"`: descarga exclusivamente `train.csv` y `data_dictionary.csv` a través de la API de la competición. También admite `"LOCAL"` si ya descargaste los archivos.
- **Autenticación única:** el notebook obtiene el token desde **Secretos de Colab** (`KAGGLE_API_TOKEN`). No solicita valores manualmente ni permite cargar credenciales dentro del cuaderno.
- `COHORTES_EVALUAR=["5_17"]` reproduce **los resultados principales** del informe. Opcionalmente agrega `"5_19"` y `"5_22"` para los análisis de sensibilidad.
- `INCLUIR_ABLACIONES=True` añade **B + sueño** y **B + internet**, solo para 5–17 años.

Para obtener un token nuevo, visita [Kaggle → Settings → API](https://www.kaggle.com/settings/api), genera **New Token** y guarda su valor en el panel de Secretos de Google Colab (ícono de llave). No es necesario generar uno nuevo si el actual sigue vigente.

La instalación intenta usar scikit-learn 1.8.0, versión registrada en el experimento original, cuando se ejecuta en Colab. Pequeñas diferencias de resultados pueden aparecer por bibliotecas y plataformas distintas.

In [ ]:
# MODIFICAR SOLO ESTAS OPCIONES (EL TOKEN SE LEE DESDE SECRETOS DE GOOGLE COLAB)
FUENTE_DATOS = "KAGGLE"  # "KAGGLE" o "LOCAL"
ARCHIVO_TRAIN_LOCAL = "/content/train.csv"
ARCHIVO_DICCIONARIO_LOCAL = "/content/data_dictionary.csv"
COHORTES_EVALUAR = ["5_17"]  # opcional: ["5_17", "5_19", "5_22"]
INCLUIR_ABLACIONES = True
DESCARGAR_ZIP_AL_FINAL = False  # True ofrece descargar solo resultados agregados en Colab
FORZAR_DESCARGA_DATOS = False

from pathlib import Path
import os, sys, platform, json, zipfile, subprocess, importlib.util, hashlib

EN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
RUTA_BASE = Path("/content" if EN_COLAB else Path.cwd())
RUTA_DATOS = RUTA_BASE / "datos_piu_miaa"
RUTA_RESULTADOS = RUTA_BASE / "resultados_piu_miaa"
RUTA_EDA = RUTA_RESULTADOS / "eda"
RUTA_MODELOS = RUTA_RESULTADOS / "modelos"
RUTA_ABLACION = RUTA_RESULTADOS / "ablaciones"
for ruta in (RUTA_DATOS,RUTA_RESULTADOS,RUTA_EDA,RUTA_MODELOS,RUTA_ABLACION):
    ruta.mkdir(parents=True,exist_ok=True)

if EN_COLAB:
    # Instalación reproducible: sin GPU, bibliotecas usadas por el protocolo original.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "kaggle>=1.7,<2", "scikit-learn==1.8.0", "statsmodels>=0.14,<0.16"],
                   check=True)

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import sklearn
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
print("Entorno:",platform.python_version(),"| pandas",pd.__version__,"| scikit-learn",sklearn.__version__)
print("Directorio de resultados:",RUTA_RESULTADOS)

## 1. Autenticación mediante Secretos de Google Colab

El cuaderno obtiene la credencial **únicamente** desde el panel **Secretos (🔑)** de [Google Colab](https://colab.research.google.com/). No solicita escribir el token en el código ni lo imprime.

1. Abre **Secretos (🔑)** en la barra izquierda de Colab y comprueba que exista `KAGGLE_API_TOKEN` (sin espacios y con mayúsculas).
2. Verifica que el **valor** contenga el token personal generado en [Kaggle → Settings → API](https://www.kaggle.com/settings/api), no el contenido de `kaggle.json`.
3. Activa el interruptor **Acceso desde cuaderno** para el secreto **en este notebook**. Si está activado, desactívalo y actívalo de nuevo; puedes volver a ejecutar la celda.
4. Ejecuta la celda siguiente. Si hay un problema, mostrará **el tipo de error y su posible solución**, sin revelar el token.

Abre el cuaderno directamente en el sitio web de Colab: algunas integraciones externas de Colab con otros editores no permiten acceder a Secretos. Debes haber aceptado también las reglas de la [competición de Kaggle](https://www.kaggle.com/competitions/child-mind-institute-problematic-internet-use).

**Seguridad:** no pegues el secreto en una celda ni lo compartas en capturas, archivos o GitHub. Si usas `FUENTE_DATOS="LOCAL"`, se omite la autenticación.


In [ ]:
# Autenticación de Kaggle: únicamente Secretos de Colab, con diagnóstico seguro.
if FUENTE_DATOS.upper() == "KAGGLE":
    if not EN_COLAB:
        raise RuntimeError(
            "Este método requiere abrir el notebook en https://colab.research.google.com/ "
            "y configurar el secreto KAGGLE_API_TOKEN."
        )

    from google.colab import userdata

    try:
        token_temporal = userdata.get("KAGGLE_API_TOKEN")
    except Exception as exc:
        # NO mostrar str(exc): basta con el tipo, sin exponer credenciales.
        tipo_error = type(exc).__name__
        if tipo_error == "SecretNotFoundError":
            instruccion = (
                "Colab no encuentra el secreto. Comprueba que el nombre sea "
                "KAGGLE_API_TOKEN (sin espacios, respetando mayúsculas)."
            )
        elif tipo_error == "NotebookAccessError":
            instruccion = (
                "El secreto existe, pero este notebook no tiene permiso. "
                "En Secretos (🔑), activa 'Acceso desde cuaderno' para KAGGLE_API_TOKEN."
            )
        elif tipo_error in ("TimeoutException", "MessageError"):
            instruccion = (
                "No se logró comunicar con el panel de Secretos. "
                "Abre este archivo directamente en el sitio web de Colab, "
                "comprueba la conexión y vuelve a conectar el entorno de ejecución."
            )
        else:
            instruccion = (
                "No se pudo consultar el panel de Secretos. Verifica el acceso "
                "desde este cuaderno y que lo ejecutas en la interfaz web de Colab."
            )
        raise RuntimeError(
            f"No se pudo leer KAGGLE_API_TOKEN ({tipo_error}). {instruccion}"
        ) from None

    if not isinstance(token_temporal, str) or not token_temporal.strip():
        raise RuntimeError(
            "KAGGLE_API_TOKEN existe, pero su valor está vacío o no es texto. "
            "Comprueba el contenido del secreto en el panel de Colab."
        )

    os.environ["KAGGLE_API_TOKEN"] = token_temporal.strip()
    del token_temporal
    print("Secreto KAGGLE_API_TOKEN leído correctamente; no se muestra su valor.")

elif FUENTE_DATOS.upper() == "LOCAL":
    print("Modo LOCAL: no es necesario acceder a Secretos.")
else:
    raise ValueError("FUENTE_DATOS debe ser KAGGLE o LOCAL.")


## 2. Descargar exclusivamente los CSV originales

La descarga usa el **comando oficial `kaggle competitions download`**, no `python -m kaggle`, para obtener únicamente `train.csv` y `data_dictionary.csv`, sin descargar los Parquet voluminosos. Se requiere haber aceptado las reglas de la competición y tener disponible el secreto `KAGGLE_API_TOKEN` en Colab.

Si ambos CSV ya están en `datos_piu_miaa/`, se reutilizan salvo que se active `FORZAR_DESCARGA_DATOS`. El notebook no sube datos ni credenciales a GitHub.


In [ ]:
COMPETICION = "child-mind-institute-problematic-internet-use"

def descargar_archivo_competicion(nombre):
    destino = RUTA_DATOS / nombre
    if destino.exists() and not FORZAR_DESCARGA_DATOS:
        return destino
    # Kaggle se ejecuta como comando de consola, no con «python -m kaggle».
    # Busca el ejecutable incluso si su directorio no aparece en PATH.
    import shutil
    import sysconfig
    ejecutable_kaggle = shutil.which("kaggle")
    if ejecutable_kaggle is None:
        candidato = Path(sysconfig.get_path("scripts")) / "kaggle"
        if candidato.is_file():
            ejecutable_kaggle = str(candidato)
    if ejecutable_kaggle is None:
        raise RuntimeError(
            "No se encontró el comando Kaggle. Ejecuta primero la celda de "
            "instalación de dependencias y, si fuese necesario, reinicia el entorno de Colab."
        )
    cmd = [ejecutable_kaggle, "competitions", "download", COMPETICION,
           "-f", nombre, "-p", str(RUTA_DATOS)]
    if FORZAR_DESCARGA_DATOS:
        cmd.append("-o")  # fuerza la descarga cuando se solicita explícitamente
    ejecucion = subprocess.run(cmd, capture_output=True, text=True)
    if ejecucion.returncode != 0:
        detalle = (ejecucion.stderr or ejecucion.stdout).strip()[-700:]
        raise RuntimeError(
            f"Falló la descarga de {nombre} desde Kaggle. "
            "Verifica que hayas aceptado las reglas de la competición y que el "
            "token tenga permisos para descargar sus datos. "
            f"Respuesta del comando: {detalle or 'sin detalles'}"
        )
    if destino.exists():
        return destino
    # Algunas versiones de la API entregan el CSV como ZIP de un único archivo.
    candidatos = sorted(RUTA_DATOS.glob("*.zip"), key=lambda p:p.stat().st_mtime, reverse=True)
    for archivo_zip in candidatos:
        with zipfile.ZipFile(archivo_zip) as z:
            coincidencias = [x for x in z.namelist() if Path(x).name == nombre]
            if coincidencias:
                destino.write_bytes(z.read(coincidencias[0]))
                return destino
    raise FileNotFoundError(f"Kaggle no devolvió {nombre}; contenido en {RUTA_DATOS}: "
                            + ", ".join(p.name for p in RUTA_DATOS.iterdir()))

if FUENTE_DATOS.upper() == "KAGGLE":
    ruta_train = descargar_archivo_competicion("train.csv")
    ruta_diccionario = descargar_archivo_competicion("data_dictionary.csv")
elif FUENTE_DATOS.upper() == "LOCAL":
    ruta_train = Path(ARCHIVO_TRAIN_LOCAL).expanduser()
    ruta_diccionario = Path(ARCHIVO_DICCIONARIO_LOCAL).expanduser()
else:
    raise ValueError("FUENTE_DATOS debe ser KAGGLE o LOCAL")
for ruta in (ruta_train,ruta_diccionario):
    if not ruta.is_file():
        raise FileNotFoundError(f"No se encuentra {ruta}")
sha256_train = hashlib.sha256(ruta_train.read_bytes()).hexdigest()
print("Datos listos:",ruta_train.name,ruta_diccionario.name)
print("Huella SHA256 de train.csv:",sha256_train)

train = pd.read_csv(ruta_train)
diccionario = pd.read_csv(ruta_diccionario)
assert train.shape == (3960,82), f"Dimensiones inesperadas: {train.shape}"
assert train["id"].is_unique and train["sii"].notna().sum()==2736
campos_prohibidos = ["id","sii"]+[x for x in train.columns if x.startswith("PCIAT-")]
variables_eda = [x for x in train.columns if x not in campos_prohibidos]
assert len(variables_eda)==58, f"Se esperaban 58 variables no PCIAT; aparecen {len(variables_eda)}"
etiquetados = train.loc[train.sii.notna()].copy()
etiquetados["sii"] = etiquetados["sii"].astype(int)
etiquetados["grupo_edad"] = pd.cut(etiquetados["Basic_Demos-Age"],[0,12,17,200],labels=["5-12","13-17","18-22"])
principal = etiquetados.loc[etiquetados["Basic_Demos-Age"].between(5,17)].copy()
assert len(principal)==2652
assert principal.sii.value_counts().reindex([0,1,2,3],fill_value=0).tolist()==[1562,700,360,30]
print("Fuente original:",train.shape,"| etiquetados:",len(etiquetados),
      "| cohorte 5–17:",len(principal),"| variables auditadas:",len(variables_eda))
print("Distribución 5–17:",principal.sii.value_counts().sort_index().to_dict())
if sha256_train!="f3d32a58301631ba3897668fc167bcc6233b005e41c35c46a75586976d0737d7":
    print("AVISO: el SHA256 difiere de la copia usada en el informe; verifica versión de datos.")

## 3. EDA — población, distribución de severidad y medidas

Este apartado conserva las medidas del Notebook 03 original sobre los **2.736 participantes etiquetados (5–22 años)**, e informa por separado la cohorte principal **5–17 años**. Se usan medianas disponibles sin imputación; el uso de internet se interpreta como **categoría ordinal declarada 0–3**, no como horas registradas. Los resultados se guardan como tablas agregadas.

In [ ]:
SALIDA = RUTA_EDA
campos_clave = ['Basic_Demos-Age','Physical-BMI','SDS-SDS_Total_T','CGAS-CGAS_Score',
  'PAQ_A-PAQ_A_Total','PAQ_C-PAQ_C_Total','PreInt_EduHx-computerinternet_hoursday',
  'Physical-HeartRate','BIA-BIA_Fat']
filas = []
for sii, grupo in etiquetados.groupby('sii'):
    r = {'sii':int(sii),'n':len(grupo),
         'porcentaje_sexo_codigo_0':round(100*(grupo['Basic_Demos-Sex']==0).mean(),2)}
    for campo in campos_clave:
        r[campo+'__n']=int(grupo[campo].notna().sum())
        r[campo+'__mediana']=float(grupo[campo].median()) if grupo[campo].notna().any() else None
    filas.append(r)
medianas=pd.DataFrame(filas)
medianas.to_csv(SALIDA/'01_medidas_por_sii.csv',index=False)

edades = pd.crosstab(etiquetados.grupo_edad,etiquetados.sii).reindex(columns=[0,1,2,3])
edades['total']=edades.sum(axis=1)
edades.to_csv(SALIDA/'02_edad_por_sii.csv')

for c,nombre in [('Basic_Demos-Sex','sexo'),('PreInt_EduHx-computerinternet_hoursday','internet')]:
    tab = pd.crosstab(etiquetados.sii,etiquetados[c],dropna=False).reindex([0,1,2,3])
    tab.to_csv(SALIDA/f'03_{nombre}_conteos.csv')
    tab.div(tab.sum(axis=1),axis=0).mul(100).round(2).to_csv(SALIDA/f'04_{nombre}_porcentajes_incluyendo_faltantes.csv')
    tab_val=pd.crosstab(etiquetados.sii,etiquetados[c]).reindex([0,1,2,3])
    tab_val.div(tab_val.sum(axis=1),axis=0).mul(100).round(2).to_csv(SALIDA/f'04_{nombre}_porcentajes_respuestas_validas.csv')

print(medianas[['sii','n','Basic_Demos-Age__mediana','SDS-SDS_Total_T__mediana','Physical-BMI__mediana']].to_string(index=False))
print('\nEdad x sii (casos):\n',edades)

## 4. EDA — valores faltantes y cobertura instrumental

La ausencia de PAQ-A y PAQ-C puede estar relacionada con edades de aplicación. No se considera que todos los datos faltantes sean aleatorios; no se descartaron participantes por ausencia de instrumentos.

In [ ]:
faltantes=[]
for c in variables_eda:
    r={'variable':c,'faltantes_global_pct':round(100*etiquetados[c].isna().mean(),2),
       'n_con_valor':int(etiquetados[c].notna().sum())}
    for sii,g in etiquetados.groupby('sii'):
        r[f'sii{sii}_faltantes_pct']=round(100*g[c].isna().mean(),2)
        r[f'sii{sii}_n_con_valor']=int(g[c].notna().sum())
    faltantes.append(r)
faltantes=pd.DataFrame(faltantes).sort_values('faltantes_global_pct',ascending=False)
faltantes.to_csv(SALIDA/'05_faltantes_por_sii.csv',index=False)

instrumentos={'Físico':[c for c in variables_eda if c.startswith('Physical-') and c!='Physical-Season'],
    'Bioimpedancia':[c for c in variables_eda if c.startswith('BIA-') and c!='BIA-Season'],
    'Sueño SDS':['SDS-SDS_Total_T','SDS-SDS_Total_Raw'],
    'PAQ-A':['PAQ_A-PAQ_A_Total'],'PAQ-C':['PAQ_C-PAQ_C_Total'],
    'Uso declarado internet':['PreInt_EduHx-computerinternet_hoursday'],
    'CGAS':['CGAS-CGAS_Score']}
coberturas=[]
for nombre,campos in instrumentos.items():
    for sii,g in etiquetados.groupby('sii'):
        coberturas.append({'instrumento':nombre,'sii':int(sii),'n':len(g),
            'con_al_menos_una_medida':int(g[campos].notna().any(axis=1).sum()),
            'faltantes_celdas_pct':round(100*g[campos].isna().to_numpy().mean(),2)})
pd.DataFrame(coberturas).to_csv(SALIDA/'06_cobertura_instrumentos.csv',index=False)
print('Promedio de celdas vacías en 58 campos no PCIAT:',round(100*etiquetados[variables_eda].isna().to_numpy().mean(),2),'%')
print(faltantes[['variable','faltantes_global_pct']].head(12).to_string(index=False))

## 5. EDA — asociaciones exploratorias y estratos etarios

Se emplea Spearman sobre pares disponibles; la corrección Benjamini–Hochberg es **descriptiva**, no un procedimiento confirmatorio de selección de predictores. Las correlaciones por edad permiten explorar confusión etaria, sin probar causalidad.

In [ ]:
asociaciones=[]
for c in variables_eda:
    if pd.api.types.is_numeric_dtype(etiquetados[c]) and etiquetados[c].notna().sum()>80 and etiquetados[c].nunique(dropna=True)>6:
        v=etiquetados[['sii',c]].dropna()
        rho,p=stats.spearmanr(v.sii,v[c])
        asociaciones.append({'variable':c,'n':len(v),'rho_spearman':float(rho),'p_exploratoria':float(p),
                             'n_sii3':int((v.sii==3).sum())})
# Agregar las ordinales de interés con menos de 7 códigos.
for c in ['PreInt_EduHx-computerinternet_hoursday','Basic_Demos-Sex']:
    v=etiquetados[['sii',c]].dropna()
    rho,p=stats.spearmanr(v.sii,v[c])
    asociaciones.append({'variable':c,'n':len(v),'rho_spearman':float(rho),'p_exploratoria':float(p),
                         'n_sii3':int((v.sii==3).sum())})
asociaciones=pd.DataFrame(asociaciones)
asociaciones['q_fdr_bh']=multipletests(asociaciones.p_exploratoria,method='fdr_bh')[1]
asociaciones.sort_values('q_fdr_bh').to_csv(SALIDA/'07_asociaciones_exploratorias.csv',index=False)

estratos=[]
for et,grupo in etiquetados.groupby('grupo_edad',observed=True):
    for c in ['PreInt_EduHx-computerinternet_hoursday','SDS-SDS_Total_T','Physical-BMI',
              'CGAS-CGAS_Score','PAQ_A-PAQ_A_Total','PAQ_C-PAQ_C_Total']:
        v=grupo[['sii',c]].dropna()
        if len(v)>=30 and v.sii.nunique()>1 and v[c].nunique()>1:
            rho,p=stats.spearmanr(v.sii,v[c])
        else:rho,p=np.nan,np.nan
        estratos.append({'edad':str(et),'variable':c,'n':len(v),'n_sii3':int((v.sii==3).sum()),
                         'rho_spearman':rho,'p_exploratoria':p})
pd.DataFrame(estratos).to_csv(SALIDA/'08_asociaciones_por_edad.csv',index=False)
print(asociaciones[asociaciones.variable.isin(['Basic_Demos-Age','SDS-SDS_Total_T','Physical-BMI',
 'PreInt_EduHx-computerinternet_hoursday','PAQ_A-PAQ_A_Total','PAQ_C-PAQ_C_Total'])][
 ['variable','n','rho_spearman','q_fdr_bh']].round(4).to_string(index=False))

## 6. EDA — gráficos de severidad, sueño e internet

Los porcentajes no comparten siempre el mismo denominador. En particular, el código de internet **no mide objetivamente tiempo de pantalla**.

In [ ]:
plt.figure(figsize=(8,4.5))
for sii in [0,1,2,3]:
    edades_s=etiquetados.loc[etiquetados.sii==sii,'Basic_Demos-Age']
    plt.hist(edades_s,bins=np.arange(5,24),alpha=.5,density=True,label=f'sii={sii} (n={len(edades_s)})')
plt.title('Edad por categoría de severidad');plt.xlabel('Edad, años');plt.ylabel('Densidad relativa');plt.legend();plt.tight_layout()
plt.savefig(SALIDA/'grafico_edad_sii.png',dpi=150);plt.show();plt.close()

r=pd.crosstab(etiquetados.sii,etiquetados['PreInt_EduHx-computerinternet_hoursday'],normalize='index').reindex([0,1,2,3]).fillna(0)
r.plot(kind='bar',stacked=True,figsize=(8,4),colormap='viridis')
plt.title('Uso declarado de internet por sii (respuestas disponibles)');plt.xlabel('sii');plt.ylabel('Proporción');plt.legend(title='Código de uso');plt.tight_layout()
plt.savefig(SALIDA/'grafico_internet_sii.png',dpi=150);plt.show();plt.close()

vals=[etiquetados.loc[etiquetados.sii==i,'SDS-SDS_Total_T'].dropna() for i in [0,1,2,3]]
plt.figure(figsize=(6,4));plt.boxplot(vals,tick_labels=['0','1','2','3'],showfliers=False)
plt.ylabel('SDS T');plt.xlabel('sii');plt.title('Puntuación de sueño SDS por sii');plt.tight_layout()
plt.savefig(SALIDA/'grafico_sds_sii.png',dpi=150);plt.show();plt.close()

## 7. Predictores, preprocesamiento y evaluación interna

**Bloques iniciales definidos antes del entrenamiento:** A (edad + sexo), B (A + IMC + FC + PA sistólica/diastólica), C (B + SDS-T + categoría ordinal de internet). Ablaciones **posteriores y exploratorias:** B + SDS-T, B + internet. En todos los bloques se aplican **dos modelos** y la misma validación cruzada repetida (5 × 3, semilla 2026), con preprocesamiento aprendido únicamente en entrenamiento. Los parámetros se mantienen como en el Notebook 04 original.

Se reemplazan únicamente los valores físicamente imposibles ≤0 de IMC, FC y PA por faltantes. No se eliminan automáticamente valores extremos por un umbral ajustado a la etiqueta.

**Nota sobre métricas:** los 15 pliegues reutilizan personas entre repeticiones. Su media y desviación estándar sirven para describir variación interna, **no son intervalos de confianza de muestras independientes**.

In [ ]:
# Misma especificación del Notebook 04 original. Ajustes limitados a las rutas de trabajo.
os.environ["MIAA_BASE"] = str(RUTA_DATOS)
os.environ["MIAA_SOURCE_CSV"] = str(ruta_train)
os.environ["MIAA_OUTPUT"] = str(RUTA_MODELOS)
os.environ["MIAA_COHORTS"] = ",".join(COHORTES_EVALUAR)

"""MIAA PIU: modelos de referencia PREESPECIFICADOS, con CV repetida, sin ajuste en validacion."""
from pathlib import Path
import json, sys, os, platform
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import cohen_kappa_score, f1_score, balanced_accuracy_score, confusion_matrix, recall_score

BASE = Path(os.environ.get('MIAA_BASE', '/mnt/data'))
SOURCE = Path(os.environ.get('MIAA_SOURCE_CSV', BASE / 'train.csv'))
OUTPUT = Path(os.environ.get('MIAA_OUTPUT', BASE / 'Modelos_referencia_PIU_resultados'))
OUTPUT.mkdir(parents=True,exist_ok=True)
SEED=2026
NUM_SPLITS=5
NUM_REPEATS=3
BLOCKS={
    'A_demografia':['Basic_Demos-Age','Basic_Demos-Sex'],
    'B_fisica':['Basic_Demos-Age','Basic_Demos-Sex','Physical-BMI','Physical-HeartRate','Physical-Systolic_BP','Physical-Diastolic_BP'],
    'C_ampliada':['Basic_Demos-Age','Basic_Demos-Sex','Physical-BMI','Physical-HeartRate','Physical-Systolic_BP','Physical-Diastolic_BP','SDS-SDS_Total_T','PreInt_EduHx-computerinternet_hoursday'],
}
COHORTS={'5_17':(5,17),'5_19':(5,19),'5_22':(5,22)}
CATEGORICAL={'Basic_Demos-Sex','PreInt_EduHx-computerinternet_hoursday'}

# Reglas fisicas deliberadamente minimas y ex-ante; no dependen de y ni de estadisticos de datos.
def basic_quality_clean(df):
    df=df.copy()
    for col in ('Physical-BMI','Physical-HeartRate','Physical-Systolic_BP','Physical-Diastolic_BP'):
        if col in df: df.loc[df[col] <= 0, col]=np.nan
    return df

def mk_pipeline(features, model_name):
    nums=[x for x in features if x not in CATEGORICAL]
    cats=[x for x in features if x in CATEGORICAL]
    numeric_pipe=Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),('scaler',StandardScaler())])
    categorical_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent',keep_empty_features=True)),('one_hot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
    transform=ColumnTransformer([('numeric',numeric_pipe,nums),('category',categorical_pipe,cats)],remainder='drop',sparse_threshold=0)
    if model_name=='Logistica_multiclase':
        model=LogisticRegression(C=1.0, class_weight='balanced',max_iter=1800,random_state=SEED)
    elif model_name=='Boosting_histogramas':
        model=HistGradientBoostingClassifier(max_iter=110,learning_rate=.07,max_leaf_nodes=10,min_samples_leaf=20,l2_regularization=1.0,random_state=SEED,class_weight='balanced',early_stopping=False)
    else: raise ValueError(model_name)
    return Pipeline([('preprocess',transform),('model',model)])

def main():
    df=pd.read_csv(SOURCE)
    assert df.shape[0]==3960, f'Esperados 3960, recibidos {len(df)}'
    assert df['id'].is_unique
    assert df['sii'].notna().sum()==2736
    assert not [c for v in BLOCKS.values() for c in v if c.startswith('PCIAT-') or c in ('sii','id')]
    for cols in BLOCKS.values(): assert set(cols).issubset(df.columns)
    df=basic_quality_clean(df)
    rows=[]; confs=[]; cohort_comp=[]; conf_aggregate=[]
    requested=os.environ.get('MIAA_COHORTS', '5_17').split(',')
    for cohort,(lo,hi) in COHORTS.items():
        if cohort not in requested: continue
        data=df.loc[df['sii'].notna() & df['Basic_Demos-Age'].between(lo,hi)].copy()
        y=data['sii'].astype(int).to_numpy()
        cohort_comp.append({'cohorte':cohort,'n':len(y),**{f'sii_{v}':int(sum(y==v)) for v in range(4)}})
        assert min(np.bincount(y, minlength=4))>=NUM_SPLITS
        splits=list(RepeatedStratifiedKFold(n_splits=NUM_SPLITS,n_repeats=NUM_REPEATS,random_state=SEED).split(data,y))
        for block,features in BLOCKS.items():
            x=data[features]
            for model_name in ('Logistica_multiclase','Boosting_histogramas'):
                for k,(train_idx,valid_idx) in enumerate(splits):
                    pipe=mk_pipeline(features,model_name)
                    pipe.fit(x.iloc[train_idx],y[train_idx])
                    predicted=pipe.predict(x.iloc[valid_idx])
                    truth=y[valid_idx]
                    cm=confusion_matrix(truth,predicted,labels=[0,1,2,3]); conf_aggregate.append({'cohorte':cohort,'bloque':block,'modelo':model_name,'repeticion':k//NUM_SPLITS+1,'fold':k%NUM_SPLITS+1,'cm':cm})
                    entry={'cohorte':cohort,'bloque':block,'modelo':model_name,'repeticion':k//NUM_SPLITS+1,'fold':k%NUM_SPLITS+1,'n_validacion':len(truth),'n_train':len(train_idx),
                           'qwk':float(cohen_kappa_score(truth,predicted,weights='quadratic',labels=[0,1,2,3])),
                           'f1_macro':float(f1_score(truth,predicted,labels=[0,1,2,3],average='macro',zero_division=0)),
                           'balanced_accuracy':float(balanced_accuracy_score(truth,predicted))}
                    for v in range(4):
                        entry[f'n_sii_{v}']=int(sum(truth==v))
                        entry[f'pred_sii_{v}']=int(sum(predicted==v))
                        entry[f'recall_sii_{v}']=float(recall_score(truth,predicted,labels=[0,1,2,3],average=None,zero_division=0)[v])
                    rows.append(entry)
    folds=pd.DataFrame(rows)
    group_cols=['cohorte','bloque','modelo']
    summary=folds.groupby(group_cols,sort=False).agg(
        n_folds=('qwk','size'),qwk_media=('qwk','mean'),qwk_sd=('qwk','std'),qwk_mediana=('qwk','median'),qwk_min=('qwk','min'),qwk_max=('qwk','max'),f1_macro_media=('f1_macro','mean'),f1_macro_sd=('f1_macro','std'),acc_balanceada_media=('balanced_accuracy','mean'),
        **{f'recall_sii_{i}_medio':(f'recall_sii_{i}','mean') for i in range(4)}
    ).reset_index()
    for gkey,group in folds.groupby(group_cols,sort=False):
        for v in range(4):
            pass
        key=dict(zip(group_cols,gkey))
        matching=[c['cm'] for c in conf_aggregate if all(c[k]==q for k,q in key.items())]
        cm=sum(matching)
        for i in range(4):
            for j in range(4): confs.append({**key,'real':i,'predicho':j,'conteo_en_3_repeticiones':int(cm[i,j])})
    folds.to_csv(OUTPUT/'01_metricas_por_fold.csv',index=False)
    summary.to_csv(OUTPUT/'02_resumen_modelos.csv',index=False)
    pd.DataFrame(confs).to_csv(OUTPUT/'03_matrices_confusion_3_repeticiones.csv',index=False)
    pd.DataFrame(cohort_comp).to_csv(OUTPUT/'04_cohortes.csv',index=False)
    assert folds.shape[0]==len(requested)*len(BLOCKS)*2*NUM_SPLITS*NUM_REPEATS
    assert folds['n_sii_3'].min()>=5
    meta={
    'fuente':'Child Mind Institute - Problematic Internet Use, train.csv', 'n_total_original':3960, 'n_con_sii':2736,
    'bloques_variables':BLOCKS,'cohortes':COHORTS,'algoritmos':['Logistica_multiclase','Boosting_histogramas'],
    'cv':{'tipo':'RepeatedStratifiedKFold','n_splits':NUM_SPLITS,'n_repeats':NUM_REPEATS,'random_state':SEED},
    'saneamiento_deterministico':'Solo valores <=0 de BMI, FC y PA se reemplazan por NaN; resto de extremos se mantienen; sin mirar sii.',
    'preprocesamiento':'Imputacion mediana + indicadores de ausencias + escalado numerico; imputacion moda + onehot categorico; fit exclusivamente en train folds',
    'modelos_hiperparametros':{'Logistica_multiclase':{'C':1,'max_iter':1800,'class_weight':'balanced'},'Boosting_histogramas':{'max_iter':110,'learning_rate':.07,'max_leaf_nodes':10,'min_samples_leaf':20,'l2_regularization':1.,'class_weight':'balanced','early_stopping':False}},
    'evaluacion':'QWK/F1-macro/media por fold; matrices acumuladas sobre 3 repeticiones, no participantes independientes.',
    'precauciones':['Sin test independiente','Hiperparametros fijados antes, no optimizados','No extrapolar a prediccion prospectiva','No se presentan intervalos de confianza independientes a partir de folds repetidos','No usar test.csv publico solapado con train.csv'],
    'python':platform.python_version(), 'pandas':pd.__version__, 'numpy':np.__version__}
    import sklearn;meta['scikit_learn']=sklearn.__version__
    (OUTPUT/'05_metodologia_y_versiones.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
    print('RESULTADOS\n',pd.DataFrame(cohort_comp).to_string(index=False));print('\n',summary[['cohorte','bloque','modelo','qwk_media','qwk_sd','f1_macro_media','recall_sii_3_medio']].round(3).to_string(index=False))
    return summary

resumen_referencia = main()

## 8. Ablaciones exploratorias: B + sueño y B + internet

Esta comparación se propuso después de observar los resultados iniciales del bloque C y, por tanto, **no es una hipótesis confirmatoria preinscrita**. Se repiten los mismos 15 pliegues de la cohorte de 5–17 años y los mismos clasificadores. Se comparan diferencias emparejadas de QWK sin calcular valores p ingenuos.

In [ ]:
from sklearn.metrics import precision_score
from sklearn.dummy import DummyClassifier

if INCLUIR_ABLACIONES:
    df_ab = basic_quality_clean(train)
    data_ab = df_ab.loc[df_ab["sii"].notna() & df_ab["Basic_Demos-Age"].between(5,17)].copy()
    y_ab = data_ab["sii"].astype(int).to_numpy()
    assert len(y_ab)==2652 and np.bincount(y_ab,minlength=4).tolist()==[1562,700,360,30]
    splits_ab = list(RepeatedStratifiedKFold(n_splits=5,n_repeats=3,random_state=2026).split(data_ab,y_ab))
    bloques_ab = {
        "B_mas_SDS": BLOCKS["B_fisica"]+["SDS-SDS_Total_T"],
        "B_mas_Internet": BLOCKS["B_fisica"]+["PreInt_EduHx-computerinternet_hoursday"]
    }
    registros_ab=[]
    matrices_ab={}
    for bloque,columnas in bloques_ab.items():
        for modelo in ("Logistica_multiclase","Boosting_histogramas"):
            cm_total = np.zeros((4,4),dtype=int)
            for i,(ind_ent,ind_val) in enumerate(splits_ab):
                clasificador = mk_pipeline(columnas,modelo)
                clasificador.fit(data_ab[columnas].iloc[ind_ent],y_ab[ind_ent])
                pred = clasificador.predict(data_ab[columnas].iloc[ind_val])
                real = y_ab[ind_val]
                cm_total += confusion_matrix(real,pred,labels=[0,1,2,3])
                registros_ab.append({
                    "cohorte":"5_17", "bloque":bloque,"modelo":modelo,
                    "repeticion": i//5 +1, "fold":i%5+1,
                    "qwk":float(cohen_kappa_score(real,pred,weights="quadratic",labels=[0,1,2,3])),
                    "f1_macro":float(f1_score(real,pred,average="macro",labels=[0,1,2,3],zero_division=0)),
                    "balanced_accuracy":float(balanced_accuracy_score(real,pred)),
                    "recall_sii_3":float(recall_score(real,pred,labels=[0,1,2,3],average=None,zero_division=0)[3])
                })
            matrices_ab[(bloque,modelo)] = cm_total
            print("Ablación finalizada:",bloque,"/",modelo)
    ab_folds = pd.DataFrame(registros_ab)
    ab_folds.to_csv(RUTA_ABLACION/"01_metricas_ablation_folds.csv",index=False)

    base_folds = pd.read_csv(RUTA_MODELOS/"01_metricas_por_fold.csv").query('cohorte=="5_17"')
    comparar = pd.concat([base_folds,ab_folds],ignore_index=True,sort=False)
    resumen_total = comparar.groupby(["bloque","modelo"],sort=False).agg(
        qwk_media=("qwk","mean"),qwk_sd=("qwk","std"),
        f1_macro_media=("f1_macro","mean"),
        recall_sii_3_medio=("recall_sii_3","mean"),n_pliegues=("qwk","size")
    ).reset_index()
    resumen_total.to_csv(RUTA_ABLACION/"02_resumen_cinco_bloques.csv",index=False)

    pares=[]
    for modelo in ("Logistica_multiclase","Boosting_histogramas"):
        ff=comparar[comparar.modelo==modelo]
        piv=ff.pivot_table(index=["repeticion","fold"],columns="bloque",values="qwk")
        for nom,prim,seg in [
            ("B - A","A_demografia","B_fisica"),
            ("C - B","B_fisica","C_ampliada"),
            ("B + SDS - B","B_fisica","B_mas_SDS"),
            ("B + Internet - B","B_fisica","B_mas_Internet")
        ]:
            for (rep,fold),delta in (piv[seg]-piv[prim]).items():
                pares.append({"modelo":modelo,"contraste":nom,"repeticion":rep,"fold":fold,"delta_qwk":float(delta)})
    pares = pd.DataFrame(pares)
    pares.to_csv(RUTA_ABLACION/"03_diferencias_pareadas_folds.csv",index=False)
    deltas = pares.groupby(["modelo","contraste"],sort=False).agg(
        diferencia_media=("delta_qwk","mean"),pliegues_positivos=("delta_qwk",lambda x:int((x>0).sum())),n=("delta_qwk","size")
    ).reset_index()
    deltas.to_csv(RUTA_ABLACION/"04_resumen_contrastes.csv",index=False)

    cm_ref = pd.read_csv(RUTA_MODELOS/"03_matrices_confusion_3_repeticiones.csv")
    precision_3=[]
    for bloque in ["A_demografia","B_fisica","C_ampliada",*bloques_ab]:
        for modelo in ("Logistica_multiclase","Boosting_histogramas"):
            if bloque in bloques_ab:
                cm = matrices_ab[(bloque,modelo)]
            else:
                v=cm_ref.query('cohorte=="5_17" and bloque==@bloque and modelo==@modelo')
                cm = np.zeros((4,4),dtype=int)
                for _,fila in v.iterrows():
                    cm[int(fila.real),int(fila.predicho)]=int(fila.conteo_en_3_repeticiones)
            precision_3.append({"bloque":bloque,"modelo":modelo,
                "precision_sii_3":float(cm[3,3]/cm[:,3].sum()) if cm[:,3].sum() else 0.0,
                "sensibilidad_sii_3":float(cm[3,3]/cm[3,:].sum()) if cm[3,:].sum() else 0.0,
                "verdaderos_positivos_3rep":int(cm[3,3]),
                "predichos_sii3_3rep":int(cm[:,3].sum()),"reales_sii3_3rep":int(cm[3,:].sum())})
    precision_3=pd.DataFrame(precision_3)
    precision_3.to_csv(RUTA_ABLACION/"05_precision_sii3_por_bloque.csv",index=False)

    # Modelo de mayoría como comparación conceptual, no equivale a CV externa.
    pred_mayoria = np.zeros_like(y_ab)
    print("Modelo de mayoría sii=0 | QWK:",cohen_kappa_score(y_ab,pred_mayoria,weights="quadratic"),
          "| F1-macro:",f1_score(y_ab,pred_mayoria,average="macro",labels=[0,1,2,3],zero_division=0))
    print("\nComparación principal (cohorte 5–17):")
    print(resumen_total[["bloque","modelo","qwk_media","f1_macro_media","recall_sii_3_medio"]].round(4).to_string(index=False))
    print("\nContrastes pareados:")
    print(deltas.round(4).to_string(index=False))
    print("\nPrecisión/sensibilidad de clase severa:")
    print(precision_3.query('bloque=="C_ampliada"').round(4).to_string(index=False))
else:
    print("Ablaciones omitidas por configuración. Se mantienen modelos de referencia A/B/C.")

## 9. Visualización y contrastación con resultados publicados

La tabla que se calcula en este notebook debe contrastarse con el informe. Valores de referencia **de la ejecución histórica**, no resultados precargados: cohorte 5–17, QWK medio (logística/boosting): **A 0,325/0,284; B 0,323/0,291; B+SDS 0,368/0,362; B+internet 0,361/0,338; C 0,405/0,388**. En C, la precisión `sii=3` fue ~6,0 %/~5,4 %.

Un resultado diferente requiere registrar las versiones de Python y scikit-learn, las transformaciones y la **huella SHA256** del CSV antes de interpretar la discrepancia.

In [ ]:
if INCLUIR_ABLACIONES:
    for metrica,archivo in [("qwk_media","comparacion_qwk.png"),("f1_macro_media","comparacion_f1_macro.png")]:
        piv = resumen_total.pivot(index="bloque",columns="modelo",values=metrica)
        ax=piv.plot(kind="bar",figsize=(9,4.6),rot=20)
        ax.set_title("Validación cruzada interna — "+metrica)
        ax.set_ylabel(metrica)
        ax.set_xlabel("Bloque de predictores")
        ax.figure.tight_layout()
        ax.figure.savefig(RUTA_ABLACION/archivo,dpi=150)
        plt.show()
        plt.close(ax.figure)

    # Control descriptivo del QWK de C: no se exige identidad exacta entre plataformas.
    esperado = {("C_ampliada","Logistica_multiclase"):0.404864,
                ("C_ampliada","Boosting_histogramas"):0.387721}
    for (b,m), valor in esperado.items():
        obtenido=float(resumen_total.query("bloque==@b and modelo==@m").qwk_media.iloc[0])
        print(f"{b} / {m}: obtenido {obtenido:.5f}; original {valor:.5f}; diferencia {obtenido-valor:+.5f}")
else:
    print(resumen_referencia[['cohorte','bloque','modelo','qwk_media','f1_macro_media']].round(4).to_string(index=False))

## 10. Trazabilidad y exportación de resultados **agregados**

Se crea un manifiesto con fuentes, tamaños, bloqueos contra fuga de información, versiones y parámetros. El ZIP no incluirá `train.csv`, `data_dictionary.csv`, identificadores, credenciales ni predicciones individuales. Las tablas de matrices acumulan **tres repeticiones de las mismas personas**, no participantes adicionales.

**Limitación esencial:** la auditoría exhaustiva de Parquet (996 series) se documentó en un análisis separado; **no se recalcula aquí** porque la decisión de no usar actigrafía en A/B/C evita descargar un volumen considerable de datos. Para reproducir esa auditoría específica haría falta el procedimiento de control temporal original. Este notebook reproduce el EDA *tabular* y los experimentos de referencia/ablación.

In [ ]:
from datetime import datetime,timezone

registro = {
    "fecha_utc":datetime.now(timezone.utc).isoformat(),
    "proyecto":"MIAA 2026-2 PIU - EDA tabular y modelos",
    "competicion":COMPETICION,
    "fuente":"train.csv original de Kaggle/HBN",
    "sha256_train_csv":sha256_train,
    "filas_train":int(train.shape[0]),
    "columnas_train":int(train.shape[1]),
    "participantes_etiquetados":int(len(etiquetados)),
    "cohorte_5_17":int(len(principal)),
    "distribucion_sii_5_17":principal.sii.value_counts().sort_index().to_dict(),
    "cantidad_campos_PCIAT":len([c for c in train if c.startswith("PCIAT-")]),
    "predictores_bloques":BLOCKS,
    "semilla":2026,"validacion":"RepeatedStratifiedKFold 5 x 3",
    "cohortes_modeladas":COHORTES_EVALUAR,
    "ablaciones_exploratorias_ejecutadas":INCLUIR_ABLACIONES,
    "limitaciones":["Sin validación externa","Temporalidad tabular no comprobada",
      "Actigrafía excluida del entrenamiento", "Precisión muy baja en severidad alta",
      "Pliegues repetidos no independientes"],
    "versiones":{"python":platform.python_version(),"numpy":np.__version__,
                 "pandas":pd.__version__,"scipy":scipy.__version__,
                 "scikit_learn":sklearn.__version__}
}
(RUTA_RESULTADOS/"manifiesto.json").write_text(json.dumps(registro,ensure_ascii=False,indent=2),encoding="utf-8")

salida_zip=RUTA_BASE/"Resultados_PIU_MIAA_Notebook_Unificado.zip"
with zipfile.ZipFile(salida_zip,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=7) as archivador:
    for archivo in sorted(RUTA_RESULTADOS.rglob("*")):
        if archivo.is_file():
            archivador.write(archivo,arcname=str(archivo.relative_to(RUTA_RESULTADOS)))
print("ZIP generado:",salida_zip,"| KB:",round(salida_zip.stat().st_size/1024,1))
print("ARCHIVOS DE SALIDA (agregados):")
for p in sorted(RUTA_RESULTADOS.rglob("*")):
    if p.is_file(): print(" -",p.relative_to(RUTA_RESULTADOS))

if DESCARGAR_ZIP_AL_FINAL and EN_COLAB:
    from google.colab import files
    files.download(str(salida_zip))

## Notas de interpretación y créditos

1. Las asociaciones descriptivas del EDA no establecen relaciones causales.
2. Los cuatro niveles de `sii` describen categorías de una escala del conjunto HBN, no diagnósticos universales.
3. Las hipótesis direccionales H1/H2 se formalizaron tras la primera ejecución y las ablaciones H3 después de observar C: estos contrastes son **exploratorios**.
4. El desbalance de `sii=3` y las diferencias de edad limitan seriamente la generalización. El `test.csv` público no se usa como evaluación externa.
5. La inferencia concurrente no demuestra que las características estén disponibles antes del desenlace.

**Proveniencia del código:** Notebook_03_EDA_sii_MIAA_Colab.ipynb; Notebook_04_Modelos_referencia_PIU_MIAA_Colab.ipynb; script de análisis metodológico del paquete Resultados_Analisis_Metodologico_Segunda_Etapa_MIAA_v1.zip. Se unificaron las fuentes para facilitar revisión académica; no se incluyeron ni descargaron credenciales reales. **Este borrador necesita ejecutarse en Colab con Kaggle para verificar reproducción completa frente a los resultados del informe.**